# When / Otherwise

- This requirement is similar to the last, but now you want to add multiple values based on the voter's position. Modify your `voter_df` DataFrame to add a random number to any voting member that is defined as a `Councilmember`. Use `2` for the `Mayor` and `0` for anything other position.

- The `voter_df` Data Frame is defined and available to you. The `pyspark.sql.functions` library is available as `F`. You can use `F.rand()` to generate the random value.

## Instructions

- Add a column to `voter_df` named `random_val` with the results of the `F.rand()` method for any voter with the title `Councilmember`. Set `random_val` to `2` for the `Mayor`. Set any other title to the value `0`.
- Show some of the Data Frame rows, noting whether the clauses worked.
- Use the `.filter` clause to find 0 in `random_val`.

In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
# If you are developing an application you should use local mode. 
# Running in yarn mode is bit time consuming than running in local mode
# If you still insist to run in yarn mode then make sure that
# 1. You assigned more than one processor in the VM settings
# 2. Run only one kernel at a time

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Demo'). \
#     master('yarn'). \
#     getOrCreate()

In [5]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Demo'). \
    master('local[*]'). \
    getOrCreate()

In [6]:
sc = spark.sparkContext

In [27]:
import pyspark.sql.functions as F

# Load the CSV file
voter_df = spark.read.format('csv').options(Header=True).load('file:///home/talentum/UBUNTU_HOME/test-jupyter/P3/M2/SM2/2_ConditionalDataFramecolumnoperations/Dataset/DallasCouncilVoters.csv.gz')

# Add a column to voter_df for a voter based on their position
voter_df = voter_df.withColumn('random_val',
                               F.when(voter_df.TITLE == 'Councilmember', F.rand())
                               .when(voter_df.TITLE == 'Mayor', 2)
                               .otherwise(0))

# Show some of the DataFrame rows
voter_df.show()

# Use the .filter() clause with random_val
voter_df.filter(voter_df.random_val==0).show()


+----------+-------------+-------------------+--------------------+
|      DATE|        TITLE|         VOTER_NAME|          random_val|
+----------+-------------+-------------------+--------------------+
|02/08/2017|Councilmember|  Jennifer S. Gates|0.011789847378098406|
|02/08/2017|Councilmember| Philip T. Kingston|  0.8719417253105868|
|02/08/2017|        Mayor|Michael S. Rawlings|                 2.0|
|02/08/2017|Councilmember|       Adam Medrano| 0.45611707574308613|
|02/08/2017|Councilmember|       Casey Thomas|  0.4569426792960286|
|02/08/2017|Councilmember|Carolyn King Arnold|  0.5031665640014702|
|02/08/2017|Councilmember|       Scott Griggs|  0.4154687755253401|
|02/08/2017|Councilmember|   B. Adam  McGough|  0.7598841969110119|
|02/08/2017|Councilmember|       Lee Kleinman|  0.6993346508261422|
|02/08/2017|Councilmember|      Sandy Greyson|  0.3411352285499536|
|02/08/2017|Councilmember|  Jennifer S. Gates|  0.7866048812794422|
|02/08/2017|Councilmember| Philip T. Kingston|  

In [58]:
voter_df = voter_df.withColumn('splits', F.split('VOTER_NAME', ' '))\
    .withColumn('First Name', 
        F.when(F.col('VOTER_NAME').rlike(r'\s+'), F.col('splits').getItem(0))
    )\
    .withColumn("last name", 
        F.when(F.size('splits') > 2, 
            F.concat_ws(' ', F.col('splits').getItem(1), F.col('splits').getItem(F.size('splits') - 1))
        ).otherwise(F.col('splits').getItem(F.size('splits') - 1))
    )\
    .drop('splits')  # This removes the intermediate column from the final DataFrame

voter_df.show()

+----------+-------------+-------------------+--------------------+----------+------------+
|      DATE|        TITLE|         VOTER_NAME|          random_val|First Name|   last name|
+----------+-------------+-------------------+--------------------+----------+------------+
|02/08/2017|Councilmember|  Jennifer S. Gates|0.011789847378098406|  Jennifer|    S. Gates|
|02/08/2017|Councilmember| Philip T. Kingston|  0.8719417253105868|    Philip| T. Kingston|
|02/08/2017|        Mayor|Michael S. Rawlings|                 2.0|   Michael| S. Rawlings|
|02/08/2017|Councilmember|       Adam Medrano| 0.45611707574308613|      Adam|     Medrano|
|02/08/2017|Councilmember|       Casey Thomas|  0.4569426792960286|     Casey|      Thomas|
|02/08/2017|Councilmember|Carolyn King Arnold|  0.5031665640014702|   Carolyn| King Arnold|
|02/08/2017|Councilmember|       Scott Griggs|  0.4154687755253401|     Scott|      Griggs|
|02/08/2017|Councilmember|   B. Adam  McGough|  0.7598841969110119|        B.|Ad